# Cross-robot visual proprioception flow

Trains visual proprioception across robots, as in Robi's multi-robot experiments (2025-10): the sensor processing (SP) is trained on the demonstrations of one robot, the regressor on those of another (or the same) robot. A pair `(sp_pack, vp_pack)` means: SP trained on `sp_pack`, regressor trained and evaluated on `vp_pack`.

Each pack names its robot (`robot_exp`, `robot_run` in the pack yaml); the propriotuned SPs and the regressors take their output size from that robot. Each SP is trained once and used by every pair that needs it. There is one comparison per pair, and a summary table at the end, next to Robi's numbers.

`robot_variant = "robi"`, `demo_order = ROBI_DEMO_ORDER` and `eval_group = "vp_validation"` reproduce Robi's protocol (his joints and limits, his split, evaluation on demonstrations that the SP was trained on). `robot_variant = "full"`, `demo_order = None` and `eval_group = "vp_testing"` give the clean protocol. See `PLAN-MultiRobot.md`.

In [ ]:
import sys
sys.path.append("..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import pandas as pd
import yaml
from flow import setup_flow, flow_entry, run_flow, display_flow_report
from demonstration.demopack import import_demopack, group_chooser_sp_vp_standard, group_chooser_fixed_order
from robot.position_factory import robot_output_size

## Parameters

In [ ]:
# The order in which Robi's machine listed the demonstrations (the same names in
# every pack). The standard group chooser applied to it gives his split.
ROBI_DEMO_ORDER = [
    "2025_10_26__03_27_03", "2025_10_24__20_17_15", "2025_10_25__02_58_05", "2025_10_25__18_59_04",
    "2025_10_24__19_30_56", "2025_10_25__21_34_30", "2025_10_24__02_18_43", "2025_10_24__01_51_54",
    "2025_10_24__01_34_46", "2025_10_25__05_32_59"]

In [ ]:
flow_name = "CrossRobot_flow_00"
# the short names of the packs
packs = {"g1": "g1-left-arm-freeform", "panda": "panda-dream-freeform",
         "hand": "inspire-hand-freeform-6dof"}
# (sp_pack, vp_pack): SP trained on sp_pack, regressor trained and evaluated on vp_pack
pairs = [["panda", "panda"], ["g1", "g1"], ["panda", "g1"], ["g1", "panda"], ["hand", "hand"]]
sp_types = ["conv_vae", "vgg19", "resnet50"]
latent_sizes = [128, 256]
epochs_sp = 300
epochs_vp = 1000
image_size = [256, 256] # for the propriotuned CNNs
camera = "dev000"
# "robi": the robot runs with Robi's joints and limits (<run>_robi); "full": all moving joints
robot_variant = "robi"
# the order of the demonstrations for the split; None for the sorted order
demo_order = ROBI_DEMO_ORDER
# the group of the vp_pack the regressors are evaluated on: vp_validation (Robi) or vp_testing
eval_group = "vp_validation"
creation_style = "exist-ok"

## Workspace, demonstrations and robots

In [ ]:
expruns_path, results_path, notebooks_path = setup_flow(flow_name, [
    "demonstration", "robot_franka", "robot_g1", "robot_inspire_hand",
    "sensorprocessing_conv_vae", "sensorprocessing_propriotuned_cnn",
    "visual_proprioception", "visual_proprioception_collections"])

if demo_order is None:
    group_chooser = group_chooser_sp_vp_standard
else:
    group_chooser = group_chooser_fixed_order(demo_order, group_chooser_sp_vp_standard)

# for every pack: its demonstrations by group, and its robot (exp, run)
selections = {}
robots = {}
for pack in sorted({pack for pair in pairs for pack in pair}):
    pack_path = pathlib.Path(Config()["demopacks_path"], packs[pack]).expanduser()
    selections[pack] = import_demopack(pack_path, group_chooser)
    exp_demo = Config().get_experiment("demonstration", packs[pack])
    robot_run = exp_demo["robot_run"] + ("_robi" if robot_variant == "robi" else "")
    robots[pack] = (exp_demo["robot_exp"], robot_run)
    print(f"{pack}: {packs[pack]}, robot {robots[pack]}")

def demo_data(pack, group):
    """The [pack, demonstration, camera] triples of a group of demonstrations of a pack"""
    return [[packs[pack], demo, camera] for demo in selections[pack][group]]

def output_size(pack):
    """The size of the normalized position vector of the robot of the pack"""
    return robot_output_size(Config().get_experiment(*robots[pack], create_data_dir=False))

## Generators of the exp/runs

In [ ]:
SENSOR_PROCESSING = {"conv_vae": "ConvVaeSensorProcessing",
                     "vgg19": "VGG19ProprioTunedSensorProcessing",
                     "resnet50": "ResNetProprioTunedSensorProcessing"}

def write_exprun(exp_name, run_name, val):
    path = pathlib.Path(Config().get_exprun_path(), exp_name, run_name + ".yaml")
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        yaml.dump(val, f)

def generate_sp(sp_type, latent_size, pack):
    """The SP of the given type and latent size, trained on the pack.
    Returns the experiment, the run and the flow entry training it."""
    run = f"sp_{sp_type}_{latent_size}_{pack}"
    val = {"latent_size": latent_size, "epochs": epochs_sp,
           "training_data": demo_data(pack, "sp_training"),
           "validation_data": demo_data(pack, "sp_validation")}
    if sp_type == "conv_vae":
        exp_name = "sensorprocessing_conv_vae"
        val["save_period"] = 5
        write_exprun(exp_name, run, val)
        return exp_name, run, flow_entry("Train_SP_Conv-VAE", exp_name, run, 0, creation_style)
    exp_name = "sensorprocessing_propriotuned_cnn"
    val.update({"image_size": image_size, "robot_exp": robots[pack][0], "robot_run": robots[pack][1],
                "output_size": output_size(pack), "batch_size": 32,
                "loss": "MSELoss", "learning_rate": 0.001,
                "class": SENSOR_PROCESSING[sp_type]})
    if sp_type == "vgg19":
        val["model"] = "VGG19ProprioTunedRegression"
    elif sp_type == "resnet50":
        val.update({"model": "ResNetProprioTunedRegression", "freeze_feature_extractor": True,
                    "reductor_step_1": 512, "proprio_step_1": 64, "proprio_step_2": 16})
    else:
        raise Exception(f"Unknown sp_type {sp_type}")
    write_exprun(exp_name, run, val)
    return exp_name, run, flow_entry("Train_SP_CNN", exp_name, run, 0, creation_style)

def generate_vp(sp_exp, sp_run, sp_type, latent_size, sp_pack, vp_pack):
    """The regressor on top of the SP (trained on sp_pack), trained and evaluated on vp_pack.
    Returns the run and the flow entries training and verifying it."""
    run = f"vp_{sp_type}_{latent_size}_{sp_pack}_to_{vp_pack}"
    val = {"name": run, "encoding_size": latent_size, "output_size": output_size(vp_pack),
           "training_data": demo_data(vp_pack, "vp_training"),
           "validation_data": demo_data(vp_pack, eval_group),
           "regressor_hidden_size_1": 64, "regressor_hidden_size_2": 64,
           "loss": "MSE", "epochs": epochs_vp, "batch_size": 64,
           "sensor_processing": SENSOR_PROCESSING[sp_type],
           "sp_experiment": sp_exp, "sp_run": sp_run,
           "robot_exp": robots[vp_pack][0], "robot_run": robots[vp_pack][1],
           "input-to-notebook": ["visual_proprioception/Train_VisualProprioception.ipynb",
                                 "visual_proprioception/Verify_VisualProprioception.ipynb"]}
    write_exprun("visual_proprioception", run, val)
    return run, [flow_entry(f"Train_{run}", "visual_proprioception", run, 0, creation_style),
                 flow_entry(f"Verify_{run}", "visual_proprioception", run, 1, "exist-ok")]

def generate_compare(sp_pack, vp_pack, vp_runs):
    """The comparison of the regressors of a pair. Returns the run and its flow entry."""
    run = f"crossrobot_{sp_pack}_to_{vp_pack}"
    val = {"name": run, "source_experiment": "visual_proprioception", "tocompare": vp_runs,
           "robot_exp": robots[vp_pack][0], "robot_run": robots[vp_pack][1],
           "input-to-notebook": ["visual_proprioception/Compare_VisualProprioception.ipynb"]}
    write_exprun("visual_proprioception_collections", run, val)
    return run, flow_entry(f"Compare_{run}", "visual_proprioception_collections", run, 0, creation_style)

## The queue: the SPs, the regressors of every pair, the comparisons

In [ ]:
expruns = []
# each SP is trained once, on its pack: (sp_type, latent_size, pack) -> (experiment, run)
sps = {}
for sp_pack in sorted({sp_pack for sp_pack, _ in pairs}):
    for sp_type in sp_types:
        for latent_size in latent_sizes:
            sp_exp, sp_run, entry = generate_sp(sp_type, latent_size, sp_pack)
            sps[(sp_type, latent_size, sp_pack)] = (sp_exp, sp_run)
            expruns.append(entry)

# (sp_pack, vp_pack) -> the comparison run of the pair
compare_runs = {}
compare_entries = []
for sp_pack, vp_pack in pairs:
    vp_runs = []
    for sp_type in sp_types:
        for latent_size in latent_sizes:
            sp_exp, sp_run = sps[(sp_type, latent_size, sp_pack)]
            run, entries = generate_vp(sp_exp, sp_run, sp_type, latent_size, sp_pack, vp_pack)
            vp_runs.append(run)
            expruns.extend(entries)
    run, entry = generate_compare(sp_pack, vp_pack, vp_runs)
    compare_runs[(sp_pack, vp_pack)] = run
    compare_entries.append(entry)
expruns.extend(compare_entries)
print(f"{len(expruns)} stages")

In [ ]:
flow_error = None
try:
    run_flow(expruns, expruns_path, results_path, notebooks_path)
except Exception as error:
    flow_error = error

## Summary: mean RMSE per pair and SP, next to Robi's numbers

The mean over the fields of the robot run of the per-field RMSE (normalized units) written by each comparison. Robi's numbers: the `all_cnns_7DOF` and `Hand_6DOF` results, mean over his six learned fields.

In [ ]:
# Robi's mean RMSE, per (sp_pack, vp_pack) and SP
ROBI_RESULTS = {
    ("panda", "panda"): {"conv_vae_128": .128, "conv_vae_256": .159, "vgg19_128": .105, "vgg19_256": .109, "resnet50_128": .167, "resnet50_256": .191},
    ("g1", "g1"):       {"conv_vae_128": .049, "conv_vae_256": .039, "vgg19_128": .016, "vgg19_256": .016, "resnet50_128": .061, "resnet50_256": .055},
    ("panda", "g1"):    {"conv_vae_128": .043, "conv_vae_256": .045, "vgg19_128": .079, "vgg19_256": .067, "resnet50_128": .114, "resnet50_256": .112},
    ("g1", "panda"):    {"conv_vae_128": .177, "conv_vae_256": .173, "vgg19_128": .191, "vgg19_256": .180, "resnet50_128": .166, "resnet50_256": .183},
    ("hand", "hand"):   {"conv_vae_128": .064, "conv_vae_256": .051, "vgg19_128": .012, "vgg19_256": .014, "resnet50_128": .135, "resnet50_256": .116},
}

if flow_error is None:
    rows = []
    for (sp_pack, vp_pack), compare_run in compare_runs.items():
        rmse = pd.read_csv(pathlib.Path(results_path, "visual_proprioception_collections",
                                        compare_run, "all_msecomparison_values.csv"), index_col=0)
        for sp_type in sp_types:
            for latent_size in latent_sizes:
                sp = f"{sp_type}_{latent_size}"
                ours = rmse[f"vp_{sp}_{sp_pack}_to_{vp_pack}"].mean()
                robi = ROBI_RESULTS.get((sp_pack, vp_pack), {}).get(sp)
                rows.append({"pair": f"{sp_pack} -> {vp_pack}", "sp": sp, "ours": ours, "robi": robi})
    summary = pd.DataFrame(rows)
    summary["difference"] = summary["ours"] - summary["robi"]
    summary.to_csv(pathlib.Path(results_path, "crossrobot_summary.csv"), index=False)
    print(summary.pivot(index="pair", columns="sp", values="ours").round(3))
    print(summary.round(3).to_string())

In [ ]:
final_results_path = pathlib.Path(results_path, "visual_proprioception_collections", compare_runs[tuple(pairs[0])])
report = display_flow_report(
    expruns, results_path, notebooks_path, final_results_path,
    flow_error=flow_error,
    preview_files=sorted(final_results_path.glob("msecomparison*.png")))
if flow_error is not None:
    raise flow_error
if not report["all-results-present"]:
    raise Exception("Flow finished without all expected stage results.")